# Agentic AI - Email assistant workflow

## 1. Introduction

In this lab, you’ll be working with an **email assistant agent**. 

This agentic workflow can carry out various tasks related to email management, including 

- sending emails, 
- searching for emails from a specific sender, 
- and deleting emails. 

You’ll give it natural language instructions - like “check unread emails from my boss” or “delete the Happy Hour email” - and see how it selects the right tools and completes the task for you.

<img src="lab_overview.png" alt="Example of a calendar assistant" width="700"/>

In [1]:
# ================================
# Imports
# ================================

from dotenv import load_dotenv
import aisuite as ai
import json
import requests
import os

# --- Local / project ---
import utils

## 2. Helper Functions for Email Tools

In [2]:
BASE_URL = "http://127.0.0.1:5000"

### Helper - API Calls and Tests

In [3]:
session = requests.Session()
session.headers.update({"User-Agent": "LF-ADP-EmailClient/1.0"})

def test_send_email():
    payload = {
        "recipient": "test@example.com",
        "subject": "Test Subject",
        "body": "This is a test email body.",
    }
    r = session.post(f"{BASE_URL}/send", json=payload)
    return utils.pretty_display("POST /send", r)

def test_list_emails():
    r = session.get(f"{BASE_URL}/emails")
    return utils.pretty_display("GET /emails", r)

def test_search_emails(q: str = "report"):
    r = session.get(f"{BASE_URL}/emails/search", params={"q": q})
    return utils.pretty_display(f"GET /emails/search?q={q}", r)

def test_filter_emails(recipient: str | None = None, date_from: str | None = None, date_to: str | None = None):
    params = {}
    if recipient:
        params["recipient"] = recipient
    if date_from:
        params["date_from"] = date_from
    if date_to:
        params["date_to"] = date_to
    r = session.get(f"{BASE_URL}/emails/filter", params=params)
    return utils.pretty_display("GET /emails/filter", r)

def test_unread_emails():
    r = session.get(f"{BASE_URL}/emails/unread")
    return utils.pretty_display("GET /emails/unread", r)

def test_get_email(email_id: str):
    r = session.get(f"{BASE_URL}/emails/{email_id}")
    return utils.pretty_display(f"GET /emails/{email_id}", r)

def test_mark_read(email_id: str):
    r = session.patch(f"{BASE_URL}/emails/{email_id}/read")
    return utils.pretty_display(f"PATCH /emails/{email_id}/read", r)

def test_mark_unread(email_id: str):
    r = session.patch(f"{BASE_URL}/emails/{email_id}/unread")
    return utils.pretty_display(f"PATCH /emails/{email_id}/unread", r)

def test_delete_email(email_id: str):
    r = session.delete(f"{BASE_URL}/emails/{email_id}")
    return utils.pretty_display(f"DELETE /emails/{email_id}", r)


### Helper - Reset Database

In [4]:
def reset_database() -> dict:
    """Calls the /reset_database endpoint and returns the confirmation message."""
    r = session.get(f"{BASE_URL}/reset_database")
    r.raise_for_status()
    return r.json()

### Helper - Email Tools

In [5]:

def list_all_emails() -> list:
    """
    Fetch all emails stored in the system, ordered from newest to oldest.

    Returns:
        List[dict]: A list of all emails including read and unread, 
        each represented as a dictionary with keys:
        - id
        - sender
        - recipient
        - subject
        - body
        - timestamp
        - read (boolean)
    """
    return requests.get(f"{BASE_URL}/emails").json()


def list_unread_emails() -> list:
    """
    Fetch all unread emails only.

    Returns:
        List[dict]: A list of unread emails (where `read == False`), 
        ordered from newest to oldest. Same structure as `list_all_emails`.
    """
    return requests.get(f"{BASE_URL}/emails/unread").json()


def search_emails(query: str) -> list:
    """
    Search emails containing the query in subject, body, or sender.

    Args:
        query (str): A keyword or phrase to search for.

    Returns:
        List[dict]: A list of emails matching the query string.
    """
    return requests.get(f"{BASE_URL}/emails/search", params={"q": query}).json()


def filter_emails(recipient: str = None, date_from: str = None, date_to: str = None) -> list:
    """
    Filter emails based on recipient and/or a date range.

    Args:
        recipient (str): Email address to filter by (optional).
        date_from (str): Start date in 'YYYY-MM-DD' format (optional).
        date_to (str): End date in 'YYYY-MM-DD' format (optional).

    Returns:
        List[dict]: A list of emails matching the given filters.
    """
    params = {}
    if recipient:
        params["recipient"] = recipient
    if date_from:
        params["date_from"] = date_from
    if date_to:
        params["date_to"] = date_to

    return requests.get(f"{BASE_URL}/emails/filter", params=params).json()


def get_email(email_id: int) -> dict:
    """
    Retrieve a specific email by its ID.

    Args:
        email_id (int): The unique ID of the email to fetch.

    Returns:
        dict: A single email record if found, else raises HTTP 404.
    """
    return requests.get(f"{BASE_URL}/emails/{email_id}").json()


def mark_email_as_read(email_id: int) -> dict:
    """
    Mark a specific email as read.

    Args:
        email_id (int): The ID of the email to mark as read.

    Returns:
        dict: The updated email record with `read: true`.
    """
    return requests.patch(f"{BASE_URL}/emails/{email_id}/read").json()


def mark_email_as_unread(email_id: int) -> dict:
    """
    Mark a specific email as unread.

    Args:
        email_id (int): The ID of the email to mark as unread.

    Returns:
        dict: The updated email record with `read: false`.
    """
    return requests.patch(f"{BASE_URL}/emails/{email_id}/unread").json()


def send_email(recipient: str, subject: str, body: str) -> dict:
    """
    Send an email (simulated). The sender is set automatically by the server.

    Args:
        recipient (str): The email address of the recipient.
        subject (str): The subject of the email.
        body (str): The message body content.

    Returns:
        dict: The created email record.
    """
    payload = {
        "recipient": recipient,
        "subject": subject,
        "body": body
    }
    return requests.post(f"{BASE_URL}/send", json=payload).json()


def delete_email(email_id: int) -> dict:
    """
    Delete an email by its ID.

    Args:
        email_id (int): The ID of the email to delete.

    Returns:
        dict: A confirmation message: {"message": "Email deleted"}
    """
    return requests.delete(f"{BASE_URL}/emails/{email_id}").json()


def search_unread_from_sender(sender: str) -> list:
    """
    Return all unread emails from a specific sender (case-insensitive match).

    Args:
        sender (str): The email address of the sender to search for.

    Returns:
        List[dict]: A list of unread emails where the sender matches the given address.
    """
    unread = list_unread_emails()
    return [e for e in unread if e['sender'].lower() == sender.lower()]


## 3. Initialize Client and Environment

In [6]:
# ================================
# Environment & Client
# ================================
load_dotenv()          # Load environment variables from .env
client = ai.Client()   # Initialize AISuite client

## 4. Simulated email service

### 4.1 Components
This lab uses a **simulated email backend** to mimic real-world email interactions.

Think of it as your personal sandbox email inbox: it comes preloaded with messages so you can practice without sending real emails.  

Instead of building this backend from scratch, you’ll be using tools that let you interact with it directly. Behind the scenes, it uses:

| Layer                   | Purpose                        |
|-------------------------|--------------------------------|
| **FastAPI**             | Exposes REST endpoints         |
| **SQLite + SQLAlchemy** | Stores and queries emails locally |
| **Pydantic**            | Ensures inputs and outputs are valid |
| **AISuite tools**       | Bridge between the LLM and the service |


### 4.2 Endpoints
The service provides several `routes` that simulate common email actions.  

Later on, these will be `wrapped as tools` so the assistant can use them automatically:

- `POST /send` → send a new email  
- `GET /emails` → list all emails  
- `GET /emails/unread` → show only unread emails  
- `GET /emails/{id}` → fetch a specific email by ID  
- `GET /emails/search?q=...` → search emails by keyword  
- `GET /emails/filter` → filter by recipient or date range  
- `PATCH /emails/{id}/read` → mark an email as read  
- `PATCH /emails/{id}/unread` → mark an email as unread  
- `DELETE /emails/{id}` → delete an email by ID  
- `GET /reset_database` → reset emails to initial state (for testing)  


> 💡 **Key idea:** These endpoints act as the “inbox controls.”  
> In the next steps, you’ll expose them as Python functions (tools) that the LLM can call—turning raw routes into agent actions.

### 4.3 Run the service

Suppose that the directory structure of your project is similar as below; 

```text
my_project/
│
├── email_assistant.ipynb  # this notebook
├── utils.py               # UI file expected in _REPO_ROOT
├── emai_tools.py          # Optional static files folder
└── email_server/          # Move your code here
    ├── __init__.py        # Empty file to make it a package
    ├── email_service.py   # The fastapi codes
    ├── email_database.py  # Database configuration
    ├── email_models.py    # SQLAlchemy models
    ├── email_schema.py    # Pydantic schemas
    └── templates/
        └── ui_all.html    # Fallback location for UI
```

Navigate to the root directory (`my_project/`) and run the application using `uvicorn`. 
Because of the service is inside a package, you should run it on command line as a module:

```bash
uvicorn email_server.main:app --reload --port 5000
```

Your Email server url would be:
```.env
EMAIL_SERVER_API_URL=http://127.0.0.1:5000
```


### 4.4 Endpoint test helpers
Before giving control to the LLM, you’ll first **test the backend yourself**.  

> 💡 **Key idea:** The `utils.test_*` functions are quick wrappers around the API endpoints. 
> They let you try actions like **send, list, search, filter, mark, and delete** without writing raw HTTP requests.


Each helper has a clear, self-explanatory name. In the next cell code, you can uncomment the one you want to run, check their output, and confirm that the email service behaves as expected.  

For example, you can test:  
- Send a test email  
- Fetch email by ID  
- List all messages  
- Mark email as read or unread  
- Delete email  

This step is your **sanity check** before handing the controls over to the agent. 

It gives you confidence that the simulated email service is alive and behaving just like a real email service would.

👉 To test the endpoints, **uncomment** the lines you’d like to try out, run the cell, and check the results instantly.

In [7]:
# uncomment the line 'utils.test_*' you want to try
new_email_id = test_send_email()
_ = test_get_email(new_email_id['id'])
# _ = test_list_emails()
# _ = test_filter_emails(recipient="test@example.com")
# _ = test_search_emails("lunch")
# _ = test_unread_emails()
# _ = test_mark_read(new_email_id['id'])
# _ = test_mark_unread(new_email_id['id'])
# _ = test_delete_email(new_email_id['id'])
# _ = reset_database()

## 5. Preparing the agent prompt

Before assigning tasks to the email assistant agent, you’ll create a small helper function called `build_prompt()`. 
This function wraps the natural language request in a system-style prompt so the LLM:

- Recognizes that it’s acting as an **email assistant agent**  
- Understands it has permission to use the available tools  
- Executes actions directly, without asking for confirmation (no human-in-the-loop)

In [8]:
def build_prompt(request_: str) -> str:
    return f"""
- You are an AI assistant specialized in managing emails.
- You can perform various actions such as listing, searching, filtering, and manipulating emails.
- Use the provided tools to interact with the email system.
- Never ask the user for confirmation before performing an action.
- If needed, my email address is "you@email.com" so you can use it to send emails or perform actions related to my account.

{request_.strip()}
"""

Run the next cell to see how the previous function **wraps your raw user prompt** with system instructions.
For example:

In [9]:
example_prompt = build_prompt("Delete the Happy Hour email")
utils.print_html(content=example_prompt, title="Example example_prompt")

Since you’ve been experimenting with the email service, let’s reset it back to its initial state.

You can do this by calling the utility function that clears and refreshes the simulated email service:

In [10]:
reset_database()

{'message': 'Database reset and emails reloaded'}

## 6. LLM + Email tools

### 6.1. Scenario 1: Normal Call

> “Check for unread emails from `boss@email.com`, mark them as read, and send a polite follow-up.”

*What happen:*
1. The agent interprets your instruction.  
2. It selects the right tools (`search_unread_from_sender` → `mark_email_as_read` → `send_email`).  
3. It executes each action automatically, without asking for confirmation.

AISuite handles schema exposure, argument binding, execution, and passing results between steps

*What to look for:*
- A clear **tool-call trace** showing which tools were used and the arguments passed.
- A concise **final message** summarizing the actions performed (e.g., “Found 1 unread email, marked as read, sent follow-up”).

In [11]:
# Try your own requests
prompt_ = build_prompt("Check for unread emails from boss@email.com, mark them as read, and send a polite follow-up.")

response = client.chat.completions.create(
    model="openai:gpt-4.1", # LLM
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[ # list of tools that the LLM can access
        search_unread_from_sender,
        list_unread_emails,
        search_emails,
        get_email,
        mark_email_as_read,
        send_email
    ],
    max_turns=5,
)

utils.pretty_print_chat_completion(response)

### 6.2. Scenario 2: Missing tool - `delete_email`

What happens if the tool you need isn’t available?  

 > “Delete alice@work.com’s email.”

Since the `delete_email` tool is not registered, the LLM will still attempt to respond, but it won’t be able to complete the action.   

> 💡 **Key idea:**  The tools you make available directly determine what the agent can do.

**Available tools**

| Tool Function                      | Action                                                                 |
|------------------------------------|------------------------------------------------------------------------|
| `list_all_emails()`                | Fetch all emails, newest first                                         |
| `list_unread_emails()`             | Retrieve only unread emails                                            |
| `search_emails(query)`             | Search by keyword in subject, body, or sender                          |
| `filter_emails(...)`               | Filter by recipient and/or date range                                  |
| `get_email(email_id)`              | Fetch a specific email by ID                                           |
| `mark_email_as_read(id)`           | Mark an email as read                                                  |
| `mark_email_as_unread(id)`         | Mark an email as unread                                                |
| `send_email(...)`                  | Send a new (mock) email                                                |
| `delete_email(id)`                 | Delete an email by ID                                                  |
| `search_unread_from_sender(addr)`  | Return unread emails from a given sender (e.g., `boss@email.com`)      |





In [16]:
# Try with a request that may call an unavailable tool
prompt_ = build_prompt("Delete alice@work.com email")

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[ # list of tools that the LLM can access
        search_unread_from_sender,
        list_unread_emails,
        search_emails,
        get_email,
        mark_email_as_read,
        send_email
    ],
    max_turns=5
)

utils.pretty_print_chat_completion(response)

**Try again with `delete_email` enabled**

The agent couldn’t delete emails because the `delete_email` tool wasn’t available.

Now add `delete_email` to the tool list and re-run the cell. 

In [17]:
prompt_ = build_prompt("Delete alice@work.com email")

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[
        search_unread_from_sender,
        list_unread_emails,
        search_emails,
        get_email,
        mark_email_as_read,
        send_email,
        delete_email
    ],
    max_turns=5
)

utils.pretty_print_chat_completion(response)

### 6.3. Scenario 3: Delete the “Happy Hour” email

The test inbox comes preloaded with a message titled **“Happy Hour.”**  Your task is to instruct the agent to locate this message and delete it.

For reference, here’s the entry from the mock dataset:

```json
{
  "id": 1,
  "sender": "eric@work.com",
  "recipient": "you@email.com",
  "subject": "Happy Hour",
  "body": "We're planning drinks this Friday!",
  "timestamp": "2025-06-13T04:48:59.096908",
  "read": false
}
````
Run the next cell and watch how the agent searches for the message and deletes it from the inbox.

In [12]:
prompt_ = build_prompt("Delete the happy hour email")

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt_
    )}],
    tools=[
        search_unread_from_sender,
        list_unread_emails,
        search_emails,
        get_email,
        mark_email_as_read,
        send_email,
        delete_email
    ],
    max_turns=5
)

utils.pretty_print_chat_completion(response)